# WESAD dataset processing

In [2]:
import os
import glob 
import wfdb
import pytz
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
import neurokit2 as nk
from dateutil import tz
from pathlib import Path
from datetime import datetime as dt
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [3]:
PHYSCIO_2017_SR = 300
WESAD_SR = 700
SWELL_SR = 2048
DOWNSAMPLE_SR = 128

# 1. Preprocessing

In [4]:
def moving_average(signal, window_size=10):
    """Compute moving average with specified window size."""
    if window_size < 1:
        raise ValueError("window_size must be >= 1")
    return np.convolve(signal, np.ones(window_size)/window_size, mode='same')

def ecg_preprocessing(signal, sample_rate, lowcut=0.5, highcut=100, ma_window=10, downsample_rate=128):
    band_passed_ecg =  nk.signal_filter(signal, sampling_rate=sample_rate, lowcut=lowcut, highcut=highcut, method='butterworth_zi', order = 2)
    emg = moving_average(band_passed_ecg, window_size=ma_window)
    downsampled_ecg = nk.signal_resample(emg, sampling_rate=sample_rate, desired_sampling_rate=downsample_rate)
    return downsampled_ecg

def z_scale(arr):
     scaler = StandardScaler()
     x = scaler.fit_transform(arr)
     return x

def minmax_scale(arr):
    scaler = MinMaxScaler()
    return scaler.fit_transform(arr)

# 2. WESAD_Dataset

In [5]:
def load_subject_pickle_data(data_dir, sub_id, data_sr, is_wesad):
    # load subject data and labels
    sub_path = os.path.join(data_dir, f"{sub_id}", f"{sub_id}.pkl") if is_wesad else os.path.join(data_dir, sub_id) 
    sub_data = pickle.load(open(sub_path, "rb"), encoding="latin1")
    labels = np.array(sub_data["label"])
    
    # Preprocessing the ECG signal
    ecg_raw = np.array(sub_data["signal"]["chest"]["ECG"][:, 0]) if is_wesad else np.array(sub_data["ECG"])
        
    cleaned_ecg = ecg_preprocessing(ecg_raw, sample_rate=data_sr, downsample_rate=DOWNSAMPLE_SR)
    #ecg_singal = z_scale(cleaned_ecg.reshape(-1, 1))
    ecg_signal = cleaned_ecg
    # assign timestamps
    start_dt = dt(2017, 11, 28, 0, 0, 0, 0, tzinfo=pytz.UTC)

    # freq = 1 / 128 seconds
    label_freq = pd.DateOffset(seconds=1 / data_sr)
    ecg_freq = pd.DateOffset(seconds=1 / DOWNSAMPLE_SR)
    label_times = pd.date_range(start=start_dt, periods=labels.shape[0], freq=label_freq, tz="UTC")
    ecg_times = pd.date_range(start=start_dt, periods=ecg_signal.shape[0], freq=ecg_freq, tz="UTC")

    #Prepare the dataframe
    label_df = pd.DataFrame({"label_sample_timestamp_utc": label_times, "y": labels})
    ECG_df = pd.DataFrame({"ecg_sample_timestamp_utc": ecg_times,"ecg": ecg_signal.flatten()})

    # align labels to ECG via merge_asof
    ECG_df = pd.merge_asof(
        ECG_df.sort_values("ecg_sample_timestamp_utc"),
        label_df.sort_values("label_sample_timestamp_utc"),
        left_on="ecg_sample_timestamp_utc",
        right_on="label_sample_timestamp_utc",
        direction="nearest",
    )

    # drop label timestamp column
    ECG_df.drop(columns="label_sample_timestamp_utc", inplace=True)
    # make timestamp index
    ECG_df.set_index("ecg_sample_timestamp_utc", inplace=True, drop=False)
    return ECG_df


In [6]:
# aggregate labels per segment (drop segments with mixed labels)
def agg_labels(label_list):
    label_set = set(label_list)
    if len(label_set) != 1:
        return np.nan
    l = list(label_set)[0]
    if l in [1,2,3]:
        return l
    return np.nan

def create_segments(ecg_df,segment_length, segment_stride):
    ecg_segs = []
    label_segs = []
    left_buffers = []
    right_buffers = []
    print(np.unique(ecg_df['y']))
    for i in range(1, len(ecg_df) - segment_length, segment_stride):
        ecg_seg = ecg_df["ecg"][i : i + segment_length]
        ecg_segs.append(list(ecg_seg))
        label_segs.append(agg_labels(ecg_df["y"][i : i + segment_length]))

        # left buffer
        if i - segment_length >= 0:
            left_buffers.append(list(ecg_df["ecg"][i - segment_length : i]))
        else:
            left_buffer = np.full_like(ecg_seg, np.nan)
            remaining_left_values = ecg_df["ecg"][:i]
            left_buffer[-remaining_left_values.shape[0] :] = remaining_left_values
            left_buffers.append(left_buffer)

        # right buffer
        if i + 2 * segment_length < len(ecg_df):
            right_buffers.append(list(ecg_df["ecg"][i + segment_length : i + 2 * segment_length]))
        else:
            right_buffer = np.full_like(ecg_seg, np.nan)
            remaining_right_values = ecg_df["ecg"][i + segment_length :]
            right_buffer[: remaining_right_values.shape[0]] = remaining_right_values
            right_buffers.append(right_buffer)
            
    keep_mask = ~np.isnan(label_segs)
    ecg_segs = np.array(ecg_segs)
    left_buffers = np.array(left_buffers)
    right_buffers = np.array(right_buffers)
    label_segs = np.array(label_segs)
    print(np.unique(label_segs, return_counts=True))
    # ---------- CREATE LABELLED PARQUET ----------
    df_labelled = pd.DataFrame({
    "x": ecg_segs[keep_mask].tolist(),  # call once on full slice
    "x_left_buffer": left_buffers[keep_mask].tolist(),
    "x_right_buffer": right_buffers[keep_mask].tolist(),
    "y": label_segs[keep_mask].tolist(),
    })
    # ---------- CREATE UNLABELLED PARQUET ----------
    df_unlabelled = pd.DataFrame({
        "x": ecg_segs.tolist(),
        "x_left_buffer": left_buffers.tolist(),
        "x_right_buffer": right_buffers.tolist(),
        "y": label_segs.tolist(),
    })
    return df_labelled, df_unlabelled


def process_subject_data(
    data_dir,
    sub_id, output_dir,
    segment_length=640, segment_stride=1, data_sr=0, is_wesad= False):
    
    ecg_df = load_subject_pickle_data(data_dir, sub_id, data_sr, is_wesad)
    # segment data
    df_labelled, df_unlabelled = create_segments(ecg_df, segment_length, segment_stride)
    print(df_labelled.shape)
    # save
    subject_out_dir = os.path.join(output_dir, sub_id)
    os.makedirs(subject_out_dir, exist_ok=True)
    df_labelled.to_parquet(os.path.join(subject_out_dir, "ECG_labelled.parquet"), index=False)
    df_unlabelled.to_parquet(os.path.join(subject_out_dir, "ECG_unlabelled.parquet"), index=False)



def load_all_wesad_subjects(data_dir, output_dir, segment_length, segment_stride):
    # run on all subjects
    subject = os.listdir(data_dir)
    subject.sort()
    for sub_id in tqdm(subject):
        if sub_id.startswith("S"):
            print("working with the subject", sub_id)
            process_subject_data(
                data_dir, sub_id, output_dir, 
                segment_length=segment_length, segment_stride=segment_stride,
                data_sr=WESAD_SR, is_wesad = True
            )


In [6]:
SEGMENT_LENGTH = 1280
SEGMENT_STRIDE = 64
WESAD_DATA_DIR = "/home/s223149341/SSL-invariance-Subject_Project_model/data/WESAD/WESAD_LOSO"
WESAD_OUTPUT_DIR = "/home/s223149341/SSL-invariance-Subject_Project_model/data/WESAD/wesad_10_05_z_scale"

load_all_wesad_subjects(WESAD_DATA_DIR, WESAD_OUTPUT_DIR,
                    SEGMENT_LENGTH, SEGMENT_STRIDE)


  0%|          | 0/16 [00:00<?, ?it/s]

working with the subject S10


  6%|▋         | 1/16 [01:28<22:11, 88.76s/it]


TypeError: float() argument must be a string or a real number, not 'Timestamp'

In [ ]:
df = pd.read_parquet('/home/s223149341/SSL-invariance-Subject_Project_model/data/WESAD/wesad_10_05/S10/ECG_labelled.parquet')
df

,x,x_left_buffer,x_right_buffer,y
0,"[0.4350993896761403, 0.4227587291043353, 0.372...","[0.5241152517057747, 0.4479350142959354, 0.355...","[0.15299997294264012, 0.21753927235356896, 0.2...",1.0
1,"[-0.8802328091755541, -0.7709747455539313, -0....","[0.700878603263332, 0.5796874848693652, 0.4838...","[0.4300915905581949, 0.4538402800476009, 0.446...",1.0
2,"[-0.6537861956846467, 0.13994133701793904, 0.3...","[0.07538467662411984, -0.054958272344688246, -...","[0.21069885628834378, 0.11922739644110023, -0....",1.0
3,"[-0.14836644122885925, -0.37025100694415203, -...","[-1.6164114646223342, -2.293234528448982, -2.8...","[-2.8641654891690442, -2.7045562997394916, -1....",1.0
4,"[0.8196932230474445, 0.7716069217221877, 0.706...","[-0.5611843283889669, -0.482108057415054, -0.3...","[0.7931183148437482, 0.648868244929643, 0.5451...",1.0
...,...,...,...,...
4491,"[-0.014481816147570374, -0.1754002456911428, 0...","[0.30232100275375684, 0.17144603232315844, 0.2...","[-0.520649439974816, -0.012640142448390863, 0....",2.0
4492,"[-0.6449719464513708, -0.28788747579703333, -0...","[-1.2859883388787248, -2.3756108799676117, -3....","[-1.348494786246596, -1.7815250144372519, -2.1...",2.0
4493,"[0.5663259897365313, 0.589498108636255, 0.7934...","[0.3302448835639161, 0.8618969265668054, 1.005...","[0.22557622292544516, 0.5913805892148948, 0.86...",2.0
4494,"[-0.12451888764832496, -0.1533932863772264, -0...","[0.5091239430400456, 0.3871063076627842, 0.277...","[-0.4752697902322011, -0.15912207005498966, -0...",2.0


## SWELL Dataset

In [7]:
# aggregate labels per segment (drop segments with mixed labels)
def agg_labels(label_list):
    label_set = set(label_list)
    if len(label_set) != 1:
        return np.nan
    l = list(label_set)[0]
    if l in [0,2,3]:
        return l
    return np.nan


In [8]:
def load_all_subjects(data_dir, output_dir, segment_length, segment_stride):
    # run on all subjects
    subject = os.listdir(data_dir)
    subject.sort()
    for sub_id in tqdm(subject):
        if sub_id.startswith("s"):
            print("working with the subject", sub_id)
            process_subject_data(
                data_dir, sub_id, output_dir, 
                segment_length=segment_length, segment_stride=segment_stride,
                data_sr=SWELL_SR, is_wesad = False
            )


In [ ]:
SEGMENT_LENGTH = 1280
SEGMENT_STRIDE = 320
SWELL_DATA_DIR = "/home/s223149341/SSL-invariance-Subject_Project_model/data/SWELL/SWELL_RAW_FULL"
SWELL_OUTPUT_DIR = "/home/s223149341/SSL-invariance-Subject_Project_model/data/SWELL/SWELL_1280_320_False_No_Relax"

load_all_subjects(SWELL_DATA_DIR, SWELL_OUTPUT_DIR,
                    SEGMENT_LENGTH, SEGMENT_STRIDE)


## Physico_net_2017


In [ ]:


def read_subject_data(hea):
    """
    Load ALL CinC2017 training ECGs as fixed-length windows.
    - root_dir: path to the 1.0.0 folder that contains 'training/'
    - preproc: callable f(x)->x (optional), e.g., baseline-wander removal
    - return_ids: if True, also return parallel list of record_ids for each window
    """
    
    rec_id = os.path.splitext(os.path.basename(hea))[0]   # "A00001"
    rec_path = os.path.join(os.path.dirname(hea), rec_id) # no extension

    sig, _ = wfdb.rdsamp(rec_path)   # works for .mat+.hea
    x = sig.squeeze().astype(np.float32)            # make 1-D if single lead
    x = ecg_preprocessing(x, sample_rate=PHYSCIO_2017_SR, downsample_rate=DOWNSAMPLE_SR)
    ecg_singal = minmax_scale(x.reshape(-1, 1))
    return ecg_singal


def create_segments_no_labels(ecg_array,segment_length, segment_stride):
    ecg_segs = []
    left_buffers = []
    right_buffers = []
    ecg_array = np.array(ecg_array)
    starts = [i for i in tqdm(range(1, len(ecg_array) - segment_length, segment_stride))]
    ecg_segs = np.stack([ecg_array[i : i + segment_length] for i in starts])

    for i in starts:
        # left buffer
        if i >= segment_length:
            left_buffers.append(ecg_array[i - segment_length : i])
        else:
            buf = np.full(segment_length, np.nan, dtype=np.float32)
            buf[-i:] = ecg_array[:i]
            left_buffers.append(buf)

        # right buffer
        if i + 2 * segment_length < len(ecg_array):
            right_buffers.append(list(ecg_array[i + segment_length : i + 2 * segment_length]))
        else:
            buf = np.full(segment_length, np.nan, dtype=np.float32)
            tail = ecg_array[i + segment_length:]
            buf[:len(tail)] = tail
            right_buffers.append(buf)
            
    # ---------- CREATE UNLABELLED PARQUET ----------
    df_unlabelled = pd.DataFrame({
        "x": ecg_segs,
        "x_left_buffer": left_buffers,
        "x_right_buffer": right_buffers,
    })

    return df_unlabelled


def process_psychio_net(
    data_dir, output_dir,
    segment_length=640, segment_stride=1):
    df_list = []
    #Load all the head path
    hea_paths = sorted(glob.glob(os.path.join(data_dir, "A*.hea")))
    subjects = sorted({fname.split(".")[0] for fname in os.listdir(data_dir)})

    for hea, s in tqdm(zip(hea_paths, subjects), total=len(subjects)):
        ecg_array = read_subject_data(hea)
        # segment data
        df_unlabelled = create_segments_no_labels(ecg_array, segment_length, segment_stride)
        df_unlabelled['subject_id'] = [s] * len(df_unlabelled)
        df_unlabelled.info
        df_list.append(df_unlabelled)

    data_df = pd.concat(df_list)
    # save
    print("Number of segments:", data_df.shape[0])
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, "physionet2017_unlabelled.parquet")

    print("Saving to:", os.path.abspath(out_path))   # <--- add this
    data_df.to_parquet(out_path, index=False)
    print("File exists after save:", os.path.exists(out_path))

        
    

: 

In [ ]:
SEGMENT_LENGTH = 1280
SEGMENT_STRIDE = 64
PSY_DATA_DIR = "/home/s223149341/SSL-invariance-Subject_Project_model/data/PhysioNet2017/raw_data"
OUTPUT_DIR = "/home/s223149341/SSL-invariance-Subject_Project_model/data/PhysioNet2017_10_5_minmax"

process_psychio_net(PSY_DATA_DIR, OUTPUT_DIR,
                    SEGMENT_LENGTH, SEGMENT_STRIDE)


100%|██████████| 40/40 [00:00<00:00, 1252.91it/s]
0it [00:00, ?it/s]2/8531 [00:14<13:11, 10.66it/s]
100%|██████████| 14/14 [00:00<00:00, 788.61it/s]
0it [00:00, ?it/s]91/8531 [01:35<11:54, 11.11it/s]
100%|██████████| 40/40 [00:00<00:00, 1005.75it/s]]
0it [00:00, ?it/s]93/8531 [02:27<21:02,  6.05it/s]
100%|██████████| 40/40 [00:00<00:00, 996.15it/s]s]
0it [00:00, ?it/s]83/8531 [02:38<19:27,  6.47it/s]
100%|██████████| 3/3 [00:00<00:00, 751.85it/s]
0it [00:00, ?it/s]048/8531 [02:56<10:37, 11.74it/s]
100%|██████████| 40/40 [00:00<00:00, 893.67it/s]/s]
0it [00:00, ?it/s]
100%|██████████| 40/40 [00:00<00:00, 620.90it/s]/s]
0it [00:00, ?it/s]178/8531 [06:01<23:12,  4.56it/s]
100%|██████████| 40/40 [00:00<00:00, 928.49it/s]
0it [00:00, ?it/s]344/8531 [06:22<12:21,  8.34it/s]
100%|██████████| 40/40 [00:00<00:00, 1090.75it/s]
0it [00:00, ?it/s]238/8531 [15:10<10:33,  6.78it/s]
 57%|█████▋    | 4831/8531 [16:38<07:08,  8.63it/s]